# 📦 Module 4.3: Model Versions

**Goal**: Create, compare, and manage multiple versions of a registered model.

---

In [ ]:
import mlflow
import mlflow.sklearn
from mlflow import MlflowClient
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score
import pandas as pd

mlflow.autolog(disable=True)
mlflow.set_experiment("04_model_versions")

wine = load_wine()
X_train, X_test, y_train, y_test = train_test_split(
    wine.data, wine.target, test_size=0.2, random_state=42
)

client = MlflowClient()
MODEL_NAME = "wine-quality-classifier"
print("✅ Setup complete!")

## 1. Create Multiple Model Versions

Each time you register a model with the same name, it creates a **new version**.

In [ ]:
# Train and register multiple model types as versions
model_configs = [
    ("rf_v1", RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42),
     {"model_type": "RandomForest", "n_estimators": 100, "max_depth": 5}),
    ("gb_v1", GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42),
     {"model_type": "GradientBoosting", "n_estimators": 100, "learning_rate": 0.1}),
    ("rf_v2", RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42),
     {"model_type": "RandomForest", "n_estimators": 200, "max_depth": 10}),
]

versions = []
for run_name, model, params in model_configs:
    with mlflow.start_run(run_name=run_name):
        mlflow.log_params(params)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", acc)
        
        # Register — creates a new version each time!
        result = mlflow.sklearn.log_model(
            model, artifact_path="model",
            registered_model_name=MODEL_NAME
        )
        
        print(f"  {run_name}: accuracy={acc:.4f} → Version {result.registered_model_version}")
        versions.append(result.registered_model_version)

print(f"\n✅ Created versions: {versions}")

## 2. List and Compare Versions

In [ ]:
# List all versions of a model
all_versions = client.search_model_versions(f"name='{MODEL_NAME}'")

print(f"📋 All versions of '{MODEL_NAME}':")
print("=" * 60)

version_data = []
for v in all_versions:
    # Get the run to fetch metrics
    try:
        run = client.get_run(v.run_id)
        accuracy = run.data.metrics.get("accuracy", None)
    except:
        accuracy = None
    
    version_data.append({
        "Version": v.version,
        "Run ID": v.run_id[:8] + "...",
        "Status": v.status,
        "Accuracy": f"{accuracy:.4f}" if accuracy else "N/A",
        "Description": v.description or "-",
        "Aliases": ", ".join(v.aliases) if v.aliases else "-"
    })

pd.DataFrame(version_data)

## 3. Add Descriptions to Versions

In [ ]:
# Add descriptions to help identify versions
if len(versions) >= 3:
    client.update_model_version(
        name=MODEL_NAME,
        version=versions[0],
        description="RandomForest baseline with 100 trees, max_depth=5"
    )
    
    client.update_model_version(
        name=MODEL_NAME,
        version=versions[1],
        description="GradientBoosting with 100 estimators, lr=0.1"
    )
    
    client.update_model_version(
        name=MODEL_NAME,
        version=versions[2],
        description="RandomForest improved with 200 trees, max_depth=10"
    )
    
    print("✅ Descriptions added to versions!")
else:
    print("Not enough versions to demonstrate. Run the cells above first.")

## 4. Add Tags to Versions

In [ ]:
# Tags help filter and organize versions
if len(versions) >= 3:
    client.set_model_version_tag(MODEL_NAME, versions[0], "validation_status", "passed")
    client.set_model_version_tag(MODEL_NAME, versions[0], "model_type", "RandomForest")
    
    client.set_model_version_tag(MODEL_NAME, versions[1], "validation_status", "passed")
    client.set_model_version_tag(MODEL_NAME, versions[1], "model_type", "GradientBoosting")
    
    client.set_model_version_tag(MODEL_NAME, versions[2], "validation_status", "pending")
    client.set_model_version_tag(MODEL_NAME, versions[2], "model_type", "RandomForest")
    
    print("✅ Tags added to versions!")

## 5. Load a Specific Version

In [ ]:
if len(versions) >= 1:
    # Load by version number
    loaded_model = mlflow.sklearn.load_model(f"models:/{MODEL_NAME}/{versions[0]}")
    
    # Make predictions
    predictions = loaded_model.predict(X_test[:5])
    
    print(f"Loaded version {versions[0]}")
    print(f"Predictions: {predictions}")
    print(f"Actual:      {y_test[:5]}")
    print(f"\n✅ Model loaded and working!")

## 🔑 Key Takeaways

1. Each registration to the same model name creates a **new version** automatically
2. Use **descriptions** to document what each version does differently
3. Use **tags** for structured metadata (validation status, model type, etc.)
4. Load by version number: `models:/model-name/3`
5. **Always track the source run** — the version links back to the original run

---

## ➡️ Next: `04_stage_transitions.ipynb` — Champion/Challenger workflow